# Practice: is model B really better?
*AI from scratch, probability 4* · lesson: https://ihor.sh/ai/p4-confidence-and-tests.html

Confidence intervals by formula and by bootstrap, a proper comparison of two models on one test set (paired test
and paired bootstrap), sample sizes for an A/B test, and peeking, simulated.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['loc is where the interval is centered (times.mean()), scale is its unit of spread (se).', 'ci_t = stats.t.interval(0.95, df=n - 1, loc=times.mean(), scale=se)'], 2: ['The middle 95% of the 10,000 resampled means leaves 2.5% below and 2.5% above.', 'ci_boot = np.percentile(boot, [2.5, 97.5])'], 3: ['Compute boolean correctness arrays ra = (pa == yte) and rb = (pb == yte).', 'Compute overall accuracies acc_a = ra.mean() and acc_b = rb.mean().', 'Count disagreement cases: b_only = int((ra & ~rb).sum()) and c_only = int((~ra & rb).sum()).', 'Compute exact binomial McNemar test p-value: stats.binomtest(c_only, b_only + c_only, 0.5).pvalue.'], 4: ['Generate 10,000 resampled index arrays: idx = rng.integers(0, len(yte), size=(10_000, len(yte))).', 'For each resample, compute accuracy difference: (pa[idx] == yte[idx]).mean(1) - (pb[idx] == yte[idx]).mean(1).', 'Compute 2.5th and 97.5th percentiles of differences: np.percentile(diffs, [2.5, 97.5]).'], 5: ['Apply rule of thumb n = round(16 * p * (1 - p) / (lift ** 2)).'], 6: ['Compute sample size n = n_per_group(0.10, 0.01).', 'Simulate 500 experiments: sample a ~ Binomial(n, 0.10)/n and b ~ Binomial(n, 0.11)/n.', 'Compute pooled standard error and z-scores for two-sample test.', 'Calculate empirical power as fraction of simulations with two-sided p < 0.05.'], 7: ['Generate 1000 simulated A/A tests (5000 users per arm, 10% conversion rate).', 'Track cumulative conversion rates at intervals of 250 visitors (looks = np.arange(250, 5001, 250)).', 'At each look, compute two-sample test z-scores and two-sided p-values.', 'Determine false_rate = (p < 0.05).any(axis=1).mean().']}
_ANSWERS = {1: 'Y2lfdCA9IHN0YXRzLnQuaW50ZXJ2YWwoMC45NSwgZGY9biAtIDEsIGxvYz10aW1lcy5tZWFuKCksIHNjYWxlPXNlKQ==', 2: 'Y2lfYm9vdCA9IG5wLnBlcmNlbnRpbGUoYm9vdCwgWzIuNSwgOTcuNV0p', 3: 'cmEsIHJiID0gcGEgPT0geXRlLCBwYiA9PSB5dGUKYWNjX2EsIGFjY19iID0gcmEubWVhbigpLCByYi5tZWFuKCkKYl9vbmx5LCBjX29ubHkgPSBpbnQoKHJhICYgfnJiKS5zdW0oKSksIGludCgofnJhICYgcmIpLnN1bSgpKQpwX21jbmVtYXIgPSBzdGF0cy5iaW5vbXRlc3QoY19vbmx5LCBiX29ubHkgKyBjX29ubHksIDAuNSkucHZhbHVl', 4: 'aWR4ID0gcm5nLmludGVnZXJzKDAsIGxlbih5dGUpLCBzaXplPSgxMF8wMDAsIGxlbih5dGUpKSkKZGlmZnMgPSAocGFbaWR4XSA9PSB5dGVbaWR4XSkubWVhbigxKSAtIChwYltpZHhdID09IHl0ZVtpZHhdKS5tZWFuKDEpCmRpZmZfY2kgPSBucC5wZXJjZW50aWxlKGRpZmZzLCBbMi41LCA5Ny41XSk=', 5: 'cmV0dXJuIHJvdW5kKDE2ICogcCAqICgxIC0gcCkgLyBsaWZ0ICoqIDIpICAgICAgICAjIHJvdW5kLCBub3QgY2VpbDogMC4xwrcwLjkvMC4wMcKyIGlzIDE0NDAwLjAwMDAwMDAwMDAwMiBpbiBmbG9hdGluZyBwb2ludA==', 6: 'biA9IG5fcGVyX2dyb3VwKDAuMTAsIDAuMDEpCmEgPSBybmcuYmlub21pYWwobiwgMC4xMCwgNTAwKSAvIG47IGIgPSBybmcuYmlub21pYWwobiwgMC4xMSwgNTAwKSAvIG4KcG9vbGVkID0gKGEgKyBiKSAvIDIKeiA9IChiIC0gYSkgLyBucC5zcXJ0KDIgKiBwb29sZWQgKiAoMSAtIHBvb2xlZCkgLyBuKQpwb3dlciA9ICgyICogc3RhdHMubm9ybS5zZihucC5hYnMoeikpIDwgMC4wNSkubWVhbigp', 7: 'QSA9IHJuZy5yYW5kb20oKDEwMDAsIDUwMDApKSA8IDAuMTsgQiA9IHJuZy5yYW5kb20oKDEwMDAsIDUwMDApKSA8IDAuMQpsb29rcyA9IG5wLmFyYW5nZSgyNTAsIDUwMDEsIDI1MCkKcmFfLCByYl8gPSBBLmN1bXN1bSgxKVs6LCBsb29rcyAtIDFdIC8gbG9va3MsIEIuY3Vtc3VtKDEpWzosIGxvb2tzIC0gMV0gLyBsb29rcwpwb29sID0gKHJhXyArIHJiXykgLyAyCnp6ID0gKHJiXyAtIHJhXykgLyBucC5zcXJ0KDIgKiBwb29sICogKDEgLSBwb29sKSAvIGxvb2tzKQpmYWxzZV9yYXRlID0gKDIgKiBzdGF0cy5ub3JtLnNmKG5wLmFicyh6eikpIDwgMC4wNSkuYW55KGF4aXM9MSkubWVhbigp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)

## 1. A confidence interval, two ways
25 measured response times (ms). First the formula way: the 95% interval for the mean from the t-distribution.
`n`, the mean and the standard error `se` are computed for you; set `ci_t` with
`stats.t.interval(0.95, df=n - 1, loc=..., scale=...)`: centered on the mean, scaled by the standard error.

In [ ]:
times = rng.lognormal(mean=4.5, sigma=0.4, size=25)
n = len(times)
se = times.std(ddof=1) / np.sqrt(n)             # the standard error of the mean
print("mean", times.mean(), " se", se)
ci_t = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(ci_t)

In [ ]:
check(1, lambda: ci_t[0] < times.mean() < ci_t[1] and abs((ci_t[1] - ci_t[0]) - 2 * stats.t.ppf(0.975, n - 1) * se) < 1e-9,
      "mean ± 2.064 standard errors (2.064 is the t value for 95% with 24 degrees of freedom)")

Now the **bootstrap** way, with no formula: resample the 25 times with replacement 10,000 times (done for you:
`boot` holds each resample's mean), and set `ci_boot` to the 2.5th and 97.5th percentiles of `boot` with `percentile`
(`np.percentile(values, [2.5, 97.5])`).

In [ ]:
boot = rng.choice(times, size=(10_000, n), replace=True).mean(axis=1)
ci_boot = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(times.mean(), ci_t, ci_boot)

In [ ]:
check(2, lambda: ci_boot[0] < times.mean() < ci_boot[1], "the bootstrap interval should contain the sample mean")
check(2, lambda: abs((ci_t[1] - ci_t[0]) - (ci_boot[1] - ci_boot[0])) < 0.3 * (ci_t[1] - ci_t[0]),
      "the two intervals should be about as wide (within 30%): two roads to the same answer")

## 2. Two models, one test set
Logistic regression vs a decision tree on the breast-cancer data. Set `acc_a`, `acc_b`, and the disagreement
counts `b_only` (A right, B wrong) and `c_only` (A wrong, B right). Then `p_mcnemar`: the exact McNemar test,
`stats.binomtest(c_only, b_only + c_only, 0.5).pvalue`.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
d = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.5, random_state=1)
pa = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr).predict(Xte)
pb = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr).predict(Xte)
acc_a = acc_b = b_only = c_only = p_mcnemar = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(acc_a, acc_b, b_only, c_only, p_mcnemar)

In [ ]:
check(3, lambda: acc_a > acc_b and b_only > c_only, 
      "Logistic regression should achieve higher accuracy and win more disagreements than depth-3 decision tree.")
check(3, lambda: p_mcnemar == stats.binomtest(c_only, b_only + c_only, 0.5).pvalue, 
      "p_mcnemar should match exact binomial test p-value stats.binomtest(c_only, b_only + c_only, 0.5).pvalue.")

## 3. A paired bootstrap for the difference
Resample **test examples** (not models) 10,000 times; for each resample compute accuracy A − accuracy B on the
same examples. Set `diff_ci`: the 95% interval of that difference. If it excludes 0, A is better on data like this.

In [ ]:
diff_ci = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("A − B:", acc_a - acc_b if acc_a is not None else None, " 95% interval:", diff_ci)

In [ ]:
check(4, lambda: diff_ci[0] < acc_a - acc_b < diff_ci[1], 
      "The 95% paired bootstrap interval should surround the point estimate of accuracy difference acc_a - acc_b.")

## 4. How many users does an A/B test need?
Write `n_per_group(p, lift)` with the rule of thumb $n \approx 16\,p(1-p)/\delta^2$ (α = 0.05, 80% power). Then
check it by simulation: with that n per group, baseline 10% and a real lift to 11%, run 500 tests and set `power`:
the share where a two-proportion z-test gives p < 0.05.

In [ ]:
def n_per_group(p, lift):
    """Estimate required sample size per variant for a two-sample proportion test (alpha=0.05, 80% power).
    
    Formula: round(16 * p * (1 - p) / (lift ** 2))
    
    Args:
        p: Baseline conversion rate (e.g. 0.10 for 10%).
        lift: Minimum detectable absolute effect size delta (e.g. 0.01 for 1% lift).
    Returns:
        int: Required sample size per arm.
    """
    # YOUR CODE HERE (stuck? hint(5), then answer(5))
    ...

power = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
print(n_per_group(0.10, 0.01), power)

In [ ]:
check(6, lambda: n_per_group(0.10, 0.01) == 14_400, 
      "Sample size per group for p=0.10, lift=0.01 should be 16 * 0.1 * 0.9 / (0.01^2) = 14,400.")
check(6, lambda: 0.72 < power < 0.88, 
      "Simulated statistical power with n=14,400 should achieve approximately 80% (between 72% and 88%).")

## 5. Peeking, simulated
1,000 A/A tests (no real difference), 5,000 visitors per arm, 10% conversion. Check after every 250 visitors and
stop at the first p < 0.05. Set `false_rate`: the share of tests that declare a winner anyway.

In [ ]:
false_rate = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print(false_rate)

In [ ]:
check(7, lambda: false_rate > 0.15, 
      "Repeated peeking (checking p-values 20 times) inflates false positive rate from 5% to over 15%.")

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 13, trees and boosting.